In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly

In [2]:
data = pd.read_csv("../data/processed/train.csv")
data_synthetic_base = pd.read_csv("../data_synthetic/synthetic_raw_1_5k.csv")
data_synthetic_few_shot = pd.read_csv("../data_synthetic/synthetic_few_shot_1_5k.csv")
data_synthetic_taxonomy_based = pd.read_csv("../data_synthetic/synthetic_taxonomy_based_1_5k.csv")
data_synthetic_decoding_params = pd.read_csv("../data_synthetic/synthetic_decoding_params_1_5k.csv")
data

,label,text,label_numeric
0,negative,жизнь сука опасная!!!!!,0
1,positive,"В моей жизни есть пара человек, переписки с ко...",2
2,neutral,"Луна зевает на тропарь,\nКомета подметает лед,...",1
3,negative,"Знаете как обидно, когда последние деньки лета...",0
4,negative,"Трио закачайся : потерянная шлюха Кирилюк, ста...",0
...,...,...,...
4644,neutral,"Покой, камин, книги, тишина... Прежде в этом в...",1
4645,neutral,В день рождения Есенина =)\n\nДа! есть горькая...,1
4646,neutral,"""сломаем обстоятельства и заново построим дом,...",1
4647,negative,Страшно - это когда удаляешь номер из телефонн...,0


## TLDR 

Synthetic data significantly differs from real data across multiple dimensions.

While class distribution is well preserved, synthetic datasets exhibit strong compression effects, reduced lexical diversity, and simplified structural patterns. In addition, synthetic generation introduces artificial artifacts in noise features and fails to reproduce realistic informal writing styles.

Importantly, despite being structurally simpler, synthetic data results in higher model uncertainty, indicating a distributional mismatch rather than reduced complexity.

These findings suggest that current LLM-based generation approaches are limited in their ability to accurately replicate the full distribution of real-world user-generated text.

## Real DATA

### LENGTH ANALYSIS

In [3]:
datasets = {
    "real": data,
    "synthetic_raw": data_synthetic_base,
    "synthetic_few_shot": data_synthetic_few_shot,
    "synthetic_taxonomy": data_synthetic_taxonomy_based,
    "synthetic_decoding": data_synthetic_decoding_params,
}

for name, df in datasets.items():
    df["char_len"] = df["text"].astype(str).apply(len)
    df["word_len"] = df["text"].astype(str).apply(lambda x: len(x.split()))

    print(f"\n{name.upper()} - WORD LEN")
    print(df["word_len"].describe())

    print(f"\n{name.upper()} - CHAR LEN")
    print(df["char_len"].describe())


REAL - WORD LEN
count    4649.000000
mean       14.118520
std        18.362749
min         1.000000
25%         4.000000
50%         8.000000
75%        15.000000
max       135.000000
Name: word_len, dtype: float64

REAL - CHAR LEN
count    4649.000000
mean       89.109486
std       115.677717
min        10.000000
25%        27.000000
50%        51.000000
75%        96.000000
max       800.000000
Name: char_len, dtype: float64

SYNTHETIC_RAW - WORD LEN
count    1500.000000
mean       11.556667
std         1.962657
min         6.000000
25%        10.000000
50%        11.000000
75%        13.000000
max        18.000000
Name: word_len, dtype: float64

SYNTHETIC_RAW - CHAR LEN
count    1500.000000
mean       72.982667
std        10.568735
min        41.000000
25%        66.000000
50%        73.000000
75%        80.000000
max       111.000000
Name: char_len, dtype: float64

SYNTHETIC_FEW_SHOT - WORD LEN
count    1500.000000
mean       11.619333
std         1.885240
min         6.000000
25%

In [48]:
import plotly.express as px
import pandas as pd

plot_df = []

for name, df in datasets.items():
    tmp = df.copy()
    tmp["source"] = name
    plot_df.append(tmp)

plot_df = pd.concat(plot_df)

fig = px.histogram(
    plot_df,
    x="word_len",
    color="source",
    nbins=50,
    barmode="overlay",
    opacity=0.5,
    title="Word Length Distribution (All Datasets)"
)
fig.show()

In [49]:
fig = px.histogram(
    plot_df,
    x="word_len",
    color="source",
    nbins=50,
    barmode="overlay",
    opacity=0.5,
    log_y=True,
    title="Word Length Distribution (log scale)"
)
fig.show()

### LENGTH ANALYSIS - Findings

- Real data is highly variable and right-skewed (mean ≈ 14, median ≈ 8, std ≈ 18)  
- A large portion of real texts are very short (25% ≤ 4 words, min = 1)  
- Real data contains a strong long tail (max = 135 words)  

- All synthetic datasets are tightly concentrated around ~11–12 words  
- Synthetic data shows very low variability:
  - std ≈ 1.8–2.0 for most methods  
  - slightly higher for decoding params (std ≈ 3.3)  

- Synthetic datasets do not contain:
  - very short texts (min ≈ 6, except decoding ≈ 3)  
  - long texts (max ≈ 17–25 vs 135 in real data)  

### Interpretation

- Real data exhibits high diversity in text length, including both short reactions and long-form messages  

- Synthetic data suffers from strong length compression:
  - most texts are generated within a narrow range  
  - extreme cases (short and long) are largely absent  

- Decoding-based generation slightly improves variability but still fails to capture the real long-tail distribution  

- This indicates a limitation of LLM-based generation:
  - tendency to produce “average-length” outputs  
  - inability to reproduce full distribution of real-world text lengths  

### CLASS DISTRIBUTION

In [50]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df["label"].value_counts(normalize=True))


REAL
label
neutral     0.512153
positive    0.253818
negative    0.234029
Name: proportion, dtype: float64

SYNTHETIC_RAW
label
neutral     0.50
positive    0.25
negative    0.25
Name: proportion, dtype: float64

SYNTHETIC_FEW_SHOT
label
neutral     0.50
negative    0.25
positive    0.25
Name: proportion, dtype: float64

SYNTHETIC_TAXONOMY
label
neutral     0.50
negative    0.25
positive    0.25
Name: proportion, dtype: float64

SYNTHETIC_DECODING
label
neutral     0.479333
negative    0.283333
positive    0.237333
Name: proportion, dtype: float64


In [51]:
plot_df = []

for name, df in datasets.items():
    tmp = df.copy()
    tmp["source"] = name
    plot_df.append(tmp)

plot_df = pd.concat(plot_df)

fig = px.histogram(
    plot_df,
    x="label",
    color="source",
    barmode="group",
    title="Class Distribution (All Datasets)"
)
fig.show()

### CLASS DISTRIBUTION - Findings

- Real data is slightly imbalanced:
  - Neutral ≈ 51%  
  - Positive ≈ 25%  
  - Negative ≈ 23%  

- Most synthetic datasets closely match this distribution:
  - Neutral ≈ 50%  
  - Positive ≈ 25%  
  - Negative ≈ 25%  

- Synthetic data with decoding parameters shows minor variation:
  - Neutral ≈ 48%  
  - Negative ≈ 28%  
  - Positive ≈ 24%  

### Interpretation

- Synthetic datasets largely replicate the class distribution of real data  

- Class balance is not a major source of discrepancy between real and synthetic datasets  

- Minor variations (e.g., in decoding-based generation) introduce small shifts but remain close to the original distribution  

- This suggests that:
  - class distribution is well controlled during synthetic generation  
  - differences between real and synthetic data are more likely to arise from other factors (e.g., text length, diversity, noise)  

### LENGTH vs CLASS 

In [53]:
plot_df = []

for name, df in datasets.items():
    tmp = df.copy()
    tmp["source"] = name
    plot_df.append(tmp)

plot_df = pd.concat(plot_df)

fig = px.box(
    plot_df,
    x="label",
    y="word_len",
    color="source",
    title="Word Length by Class (All Datasets)"
)
fig.show()

In [54]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.groupby("label")["word_len"].describe())


REAL
           count       mean        std  min  25%   50%   75%    max
label                                                              
negative  1088.0  17.309743  20.448572  1.0  6.0  10.0  20.0  133.0
neutral   2381.0  13.710206  18.684983  1.0  4.0   8.0  14.0  135.0
positive  1180.0  12.000000  14.993354  1.0  4.0   7.0  14.0  117.0

SYNTHETIC_RAW
          count       mean       std  min   25%   50%   75%   max
label                                                            
negative  375.0  11.794667  1.814373  6.0  11.0  12.0  13.0  17.0
neutral   750.0  10.820000  1.727304  6.0  10.0  11.0  12.0  18.0
positive  375.0  12.792000  1.869954  8.0  12.0  13.0  14.0  18.0

SYNTHETIC_FEW_SHOT
          count       mean       std  min   25%   50%   75%   max
label                                                            
negative  375.0  11.880000  1.590076  8.0  11.0  12.0  13.0  17.0
neutral   750.0  10.906667  1.771856  6.0  10.0  11.0  12.0  18.0
positive  375.0  12.78400

### LENGTH vs CLASS - Findings

- In real data:
  - Negative texts are the longest (mean ≈ 17.3)  
  - Neutral texts are shorter (mean ≈ 13.7)  
  - Positive texts are the shortest (mean ≈ 12.0)  
  - All classes exhibit high variability and long-tail behavior (max up to ~135, std ≈ 15–20)  

- In synthetic datasets (raw, few-shot, taxonomy):
  - All classes are tightly concentrated around ~10–13 words  
  - Very low variability (std ≈ 1.5–1.9)  
  - No long-tail behavior (max ≈ 16–18)  
  - Differences between classes are minimal and compressed  

- Synthetic data with decoding parameters:
  - Slightly higher variability (std ≈ 2.6–3.5)  
  - Some increase in length differences between classes  
  - Still far from real data variability and tail behavior  

### Interpretation

- Real data shows clear structural differences between classes:
  - Negative texts are longer and more variable  
  - Positive texts are shorter and more concise  

- Standard synthetic generation fails to capture these class-specific patterns:
  - text lengths are homogenized across classes  
  - variability is significantly reduced  

- Decoding-based generation partially improves diversity but still does not reproduce:
  - long-tail distributions  
  - strong differences between classes  

- This suggests that synthetic data lacks:
  - class-specific structural characteristics  
  - realistic variability in text length  

### LEXICAL DIVERSITY

In [55]:
def compute_lexical_stats(df):
    texts = df["text"].astype(str).tolist()
    tokens = [t.lower().split() for t in texts]
    all_tokens = [w for t in tokens for w in t]

    distinct_1 = len(set(all_tokens)) / len(all_tokens)

    bigrams = []
    for t in tokens:
        bigrams.extend(list(zip(t, t[1:])))

    distinct_2 = len(set(bigrams)) / len(bigrams) if len(bigrams) > 0 else 0

    vocab_size = len(set(all_tokens))
    total_tokens = len(all_tokens)

    return distinct_1, distinct_2, vocab_size, total_tokens


for name, df in datasets.items():
    d1, d2, vocab, total = compute_lexical_stats(df)

    print(f"\n{name.upper()}")
    print(f"distinct_1: {d1}")
    print(f"distinct_2: {d2}")
    print(f"vocab_size: {vocab}")
    print(f"total_tokens: {total}")


REAL
distinct_1: 0.3944421591480415
distinct_2: 0.8536269430051814
vocab_size: 25890
total_tokens: 65637

SYNTHETIC_RAW
distinct_1: 0.23732333429477934
distinct_2: 0.6404799494790022
vocab_size: 4114
total_tokens: 17335

SYNTHETIC_FEW_SHOT
distinct_1: 0.22439612140685064
distinct_2: 0.605436625023542
vocab_size: 3911
total_tokens: 17429

SYNTHETIC_TAXONOMY
distinct_1: 0.2720042107725598
distinct_2: 0.6892749535226617
vocab_size: 4651
total_tokens: 17099

SYNTHETIC_DECODING
distinct_1: 0.23787984226397588
distinct_2: 0.6330030487804879
vocab_size: 4102
total_tokens: 17244


### LEXICAL DIVERSITY - Findings

- Real data shows high lexical diversity:
  - Distinct-1 ≈ 0.39  
  - Distinct-2 ≈ 0.85  
  - Vocabulary size ≈ 25,890  

- All synthetic datasets have significantly lower diversity:
  - Distinct-1 ≈ 0.22–0.27  
  - Distinct-2 ≈ 0.60–0.69  
  - Vocabulary size ≈ 3,900–4,600  

- Taxonomy-based generation performs slightly better than other methods:
  - highest Distinct-1 ≈ 0.27  
  - highest Distinct-2 ≈ 0.69  

- Synthetic datasets are also much smaller in vocabulary despite comparable token counts  

### Interpretation

- Real data exhibits rich and diverse language usage with a large vocabulary and varied phrasing  

- Synthetic data shows clear lexical compression:
  - reduced vocabulary size  
  - lower diversity of word usage  
  - more repetitive patterns  

- Even the best-performing method (taxonomy-based) fails to match real data diversity  

- This indicates that synthetic generation:
  - struggles to reproduce long-tail vocabulary  
  - tends to reuse common tokens and phrases  
  - produces less linguistically diverse outputs  

### CONFIDENCE / AMBIGUITY

In [56]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

model_name = "DeepPavlov/rubert-base-cased"
model_path = "../mlartifacts/experiments_07_03_logs_RuBERT Baseline Natural Distribution_best_model.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [59]:
from tqdm import tqdm
from torch.utils.data import DataLoader

class SimpleDataset:
    def __init__(self, texts, tokenizer):
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in enc.items()}
    
def compute_confidence(df):
    dataset = SimpleDataset(df["text"].tolist(), tokenizer)
    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    all_probs = []

    with torch.no_grad():
        for batch in tqdm(loader):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            probs = torch.softmax(outputs.logits, dim=1)

            all_probs.append(probs.cpu())

    probs = torch.cat(all_probs).numpy()
    confidence = probs.max(axis=1)

    df["confidence"] = confidence

    return df


for name, df in datasets.items():
    print(f"Processing: {name}")
    datasets[name] = compute_confidence(df)

Processing: real


100%|██████████| 146/146 [01:26<00:00,  1.69it/s]


Processing: synthetic_raw


100%|██████████| 47/47 [00:29<00:00,  1.58it/s]


Processing: synthetic_few_shot


100%|██████████| 47/47 [00:29<00:00,  1.57it/s]


Processing: synthetic_taxonomy


100%|██████████| 47/47 [00:29<00:00,  1.59it/s]


Processing: synthetic_decoding


100%|██████████| 47/47 [00:28<00:00,  1.65it/s]


In [60]:
plot_df = []

for name, df in datasets.items():
    tmp = df.copy()
    tmp["source"] = name
    plot_df.append(tmp)

plot_df = pd.concat(plot_df)

fig = px.histogram(
    plot_df,
    x="confidence",
    color="source",
    nbins=50,
    barmode="overlay",
    opacity=0.5,
    title="Confidence Distribution (All Datasets)"
)
fig.show()

In [61]:
for name, df in datasets.items():
    ratio = (df["confidence"] < 0.6).mean()
    print(f"{name}: {ratio}")

real: 0.01699290169929017
synthetic_raw: 0.048
synthetic_few_shot: 0.03133333333333333
synthetic_taxonomy: 0.04533333333333334
synthetic_decoding: 0.050666666666666665


### CONFIDENCE / AMBIGUITY - Findings

- Real data shows a very low proportion of low-confidence samples (~1.7%)  

- Synthetic datasets exhibit a higher proportion of low-confidence samples:
  - Raw ≈ 4.8%  
  - Few-shot ≈ 3.1%  
  - Taxonomy ≈ 4.5%  
  - Decoding ≈ 5.1%  

- Overall, all datasets have high confidence scores, but synthetic data consistently shows more low-confidence cases  

### Interpretation

- Confidence is computed using a model trained on real data, which leads to overestimated confidence on the training set  

- Despite this, synthetic datasets produce more low-confidence predictions:
  - indicating that synthetic samples are less aligned with the patterns learned from real data  

- This suggests that synthetic data:
  - introduces distributional shifts relative to real data  
  - may contain unnatural or less coherent patterns  

- Differences between generation methods:
  - Few-shot performs closest to real data  
  - Decoding-based generation introduces the highest level of uncertainty  

- Overall, synthetic data appears less predictable for the model, despite being structurally simpler in other aspects (e.g., length, diversity)  

### CONFIDENCE vs CLASS

In [63]:
plot_df = []

for name, df in datasets.items():
    tmp = df.copy()
    tmp["source"] = name
    plot_df.append(tmp)

plot_df = pd.concat(plot_df)

fig = px.box(
    plot_df,
    x="label",
    y="confidence",
    color="source",
    title="Confidence by Class (All Datasets)"
)
fig.show()

In [64]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.groupby("label")["confidence"].agg(["mean", "std", "min", "max"]))


REAL
              mean       std       min       max
label                                           
negative  0.881007  0.096068  0.475217  0.988414
neutral   0.944307  0.070144  0.476986  0.988370
positive  0.947970  0.088412  0.410933  0.994329

SYNTHETIC_RAW
              mean       std       min       max
label                                           
negative  0.802021  0.143128  0.395770  0.990169
neutral   0.937019  0.093163  0.477752  0.987838
positive  0.954476  0.086401  0.484951  0.993590

SYNTHETIC_FEW_SHOT
              mean       std       min       max
label                                           
negative  0.848335  0.129662  0.377777  0.981839
neutral   0.947726  0.081413  0.497448  0.988558
positive  0.972327  0.064778  0.404759  0.993704

SYNTHETIC_TAXONOMY
              mean       std       min       max
label                                           
negative  0.818756  0.140896  0.478375  0.983788
neutral   0.944027  0.075522  0.503683  0.987762
positive

In [65]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.groupby("label")["confidence"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).unstack())


REAL
              0.10      0.25      0.50      0.75      0.90
label                                                     
negative  0.749809  0.865376  0.917043  0.940275  0.950714
neutral   0.890796  0.945140  0.968580  0.977964  0.982446
positive  0.866380  0.956914  0.982025  0.989379  0.991772

SYNTHETIC_RAW
              0.10      0.25      0.50      0.75      0.90
label                                                     
negative  0.578160  0.704044  0.846298  0.921965  0.947252
neutral   0.828694  0.949050  0.972800  0.979985  0.983038
positive  0.881792  0.969438  0.986211  0.991143  0.992613

SYNTHETIC_FEW_SHOT
              0.10      0.25      0.50      0.75      0.90
label                                                     
negative  0.637382  0.801628  0.907431  0.941193  0.950048
neutral   0.881841  0.958639  0.976183  0.982757  0.986065
positive  0.948498  0.984050  0.990277  0.992273  0.992919

SYNTHETIC_TAXONOMY
              0.10      0.25      0.50      0.75      

### CONFIDENCE vs CLASS - Findings

- In real data:
  - Negative class has the lowest confidence (mean ≈ 0.88) and highest variability  
  - Neutral and positive classes have higher confidence (mean ≈ 0.94–0.95)  
  - Lower quantiles confirm this gap (e.g., 10%: negative ≈ 0.75 vs neutral ≈ 0.89)  

- In synthetic datasets (raw, few-shot, taxonomy):
  - The same general pattern is preserved:
    - negative < neutral < positive in confidence  
  - However, confidence for negative class is significantly lower than in real data:
    - raw ≈ 0.80, taxonomy ≈ 0.82, few-shot ≈ 0.85  
  - Variability for negative class is also higher than in real data  

- Synthetic data with decoding parameters shows a different pattern:
  - Negative class becomes the most confident (mean ≈ 0.93)  
  - Neutral class becomes the least confident (mean ≈ 0.86)  
  - This reverses the structure observed in real data  

### Interpretation

- Real data exhibits a clear class-dependent confidence structure:
  - negative samples are more complex and less predictable  
  - neutral and positive samples are easier for the model  

- Standard synthetic generation partially preserves this structure but exaggerates uncertainty for negative samples  

- Decoding-based generation breaks this pattern:
  - alters class-specific difficulty relationships  
  - produces a distribution that does not align with real data  

- This suggests that synthetic data may distort class-level characteristics:
  - either amplifying or altering class difficulty  
  - leading to inconsistencies in how different sentiment classes are represented  

### NOISE FEATURES

In [71]:
import re

def compute_noise(df):
    df["has_emoji"] = df["text"].apply(lambda x: bool(re.search(r"[😀-🙏]", str(x))))
    df["has_caps"] = df["text"].apply(lambda x: any(w.isupper() and len(w) > 2 for w in str(x).split()))
    df["has_exclaim"] = df["text"].apply(lambda x: "!" in str(x))
    df["has_question"] = df["text"].apply(lambda x: "?" in str(x))
    df["has_multi_punct"] = df["text"].apply(lambda x: "!!" in str(x) or "??" in str(x))

    return {
        "emoji": df["has_emoji"].mean(),
        "caps": df["has_caps"].mean(),
        "exclaim": df["has_exclaim"].mean(),
        "question": df["has_question"].mean(),
        "multi_punct": df["has_multi_punct"].mean(),
    }


noise_results = {}

for name, df in datasets.items():
    noise_results[name] = compute_noise(df)

for name, stats in noise_results.items():
    print(f"\n{name.upper()}")
    for k, v in stats.items():
        print(f"{k}: {float(v):.3f}")


REAL
emoji: 0.028
caps: 0.098
exclaim: 0.284
question: 0.118
multi_punct: 0.130

SYNTHETIC_RAW
emoji: 0.254
caps: 0.004
exclaim: 0.039
question: 0.019
multi_punct: 0.001

SYNTHETIC_FEW_SHOT
emoji: 0.080
caps: 0.001
exclaim: 0.131
question: 0.155
multi_punct: 0.021

SYNTHETIC_TAXONOMY
emoji: 0.166
caps: 0.009
exclaim: 0.139
question: 0.025
multi_punct: 0.007

SYNTHETIC_DECODING
emoji: 0.217
caps: 0.011
exclaim: 0.323
question: 0.035
multi_punct: 0.323


In [70]:
rows = []

for name, stats in noise_results.items():
    for k, v in stats.items():
        rows.append({
            "source": name,
            "feature": k,
            "ratio": v
        })

noise_df = pd.DataFrame(rows)

fig = px.bar(
    noise_df,
    x="feature",
    y="ratio",
    color="source",
    barmode="group",
    title="Noise Features Distribution (All Datasets)"
)
fig.show()

### NOISE FEATURES - Findings

- Real data contains moderate levels of noise:
  - Emoji ≈ 2.8%  
  - CAPS ≈ 9.8%  
  - Exclamation marks ≈ 28.4%  
  - Question marks ≈ 11.8%  
  - Repeated punctuation ≈ 13.0%  

- Synthetic datasets show significantly different noise patterns:

  - Synthetic (raw):
    - Extremely high emoji usage (≈ 25%)  
    - Almost no CAPS (≈ 0.4%)  
    - Very low punctuation usage  

  - Synthetic (few-shot):
    - Moderate emoji usage (≈ 8%)  
    - Higher question mark usage (≈ 15.5%)  
    - Still very low CAPS and repeated punctuation  

  - Synthetic (taxonomy):
    - Elevated emoji usage (≈ 16.6%)  
    - Moderate exclamation usage (≈ 13.9%)  
    - Low CAPS and repeated punctuation  

  - Synthetic (decoding):
    - High emoji usage (≈ 21.7%)  
    - Very high exclamation (≈ 32.3%)  
    - Extremely high repeated punctuation (≈ 32.3%)  

### Interpretation

- Real data exhibits a balanced and natural distribution of noise features:
  - moderate use of punctuation  
  - noticeable presence of CAPS  
  - relatively low emoji usage  

- Synthetic data fails to reproduce realistic noise patterns:
  - CAPS usage is almost entirely missing across all methods  
  - emoji usage is significantly overrepresented  
  - punctuation usage is either suppressed or exaggerated depending on the method  

- Different generation strategies introduce distinct artifacts:
  - raw → overuse of emoji  
  - few-shot → increased questioning style  
  - taxonomy → moderate but still unrealistic noise  
  - decoding → extreme punctuation patterns  

- This indicates that synthetic data:
  - does not accurately capture informal writing styles  
  - introduces artificial stylistic biases  
  - may distort linguistic signals important for sentiment modeling  

### STRUCTURE

In [74]:
def compute_structure(df):
    df["sent_count"] = df["text"].astype(str).apply(lambda x: len(re.split(r"[.!?]", x)))
    df["avg_sent_len"] = df["word_len"] / df["sent_count"]
    return df


for name, df in datasets.items():
    datasets[name] = compute_structure(df)

In [75]:
plot_df = []

for name, df in datasets.items():
    tmp = df.copy()
    tmp["source"] = name
    plot_df.append(tmp)

plot_df = pd.concat(plot_df)

px.histogram(plot_df, x="sent_count", color="source", barmode="overlay",
             nbins=30, title="Sentence Count (All Datasets)").show()

px.histogram(plot_df, x="avg_sent_len", color="source", barmode="overlay",
             nbins=30, title="Avg Sentence Length (All Datasets)").show()

In [76]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df[["sent_count", "avg_sent_len"]].describe())


REAL
        sent_count  avg_sent_len
count  4649.000000   4649.000000
mean      3.929447      4.643199
std       4.653364      5.174306
min       1.000000      0.076923
25%       1.000000      1.750000
50%       3.000000      3.333333
75%       5.000000      6.000000
max     148.000000    135.000000

SYNTHETIC_RAW
        sent_count  avg_sent_len
count  1500.000000   1500.000000
mean      1.402000      9.954722
std       0.858421      3.465234
min       1.000000      1.400000
25%       1.000000      8.000000
50%       1.000000     11.000000
75%       1.000000     12.000000
max       5.000000     18.000000

SYNTHETIC_FEW_SHOT
        sent_count  avg_sent_len
count  1500.000000   1500.000000
mean      1.955333      8.001622
std       1.225155      3.923793
min       1.000000      1.571429
25%       1.000000      4.500000
50%       2.000000      7.500000
75%       2.000000     11.000000
max       7.000000     18.000000

SYNTHETIC_TAXONOMY
        sent_count  avg_sent_len
count  1500.000

### TEXT STRUCTURE - Findings

- Real data:
  - Average ≈ 3.9 sentences per text (median = 3)  
  - High variability (std ≈ 4.65, max = 148)  
  - 25% of texts contain only 1 sentence  
  - Average sentence length ≈ 4.6 words  

- Synthetic datasets (raw, few-shot, taxonomy):
  - Strongly biased toward single-sentence texts:
    - mean ≈ 1.4–2.0 sentences  
    - median = 1–2 sentences  
  - Low variability (max ≈ 5–7 sentences)  
  - Longer sentences (avg ≈ 8–10 words per sentence)  

- Synthetic data with decoding parameters:
  - Closest to real data in sentence count:
    - mean ≈ 3.7 sentences  
  - Still lower variability (max ≈ 11 vs 148 in real)  
  - Sentence length distribution closer to real (avg ≈ 4.3 words)  

### Interpretation

- Real data exhibits diverse and irregular structure:
  - mixture of short and long texts  
  - fragmented multi-sentence narratives  
  - high variability in both sentence count and length  

- Standard synthetic generation produces structurally simplified texts:
  - predominantly single-sentence outputs  
  - longer and more uniform sentences  
  - limited variability  

- Decoding-based generation partially recovers realistic structure:
  - better sentence segmentation  
  - closer average statistics  

- However, all synthetic methods fail to reproduce:
  - extreme cases (very long multi-sentence texts)  
  - high structural variability observed in real data  

- This indicates that synthetic data:
  - lacks structural diversity  
  - tends toward simplified and regular sentence patterns  